<a href="https://colab.research.google.com/github/MarkChernov123/ds2002-fa26/blob/main/Copy_of_2026_10_09_%E2%80%94_Public_API_to_Clean_Table_%E2%80%94_Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Public API to Clean Table

**Lab — 2026-10-09 · Fall 2026**  

---

## Lab 07 — Public API to Clean Table

Build the weather-ingestion function you will actually reuse on the midterm. It has to pull real data, survive a flaky network, say plainly when it fell back to sample data, and hand back a table that is safe to join.

The last requirement is the one people skip. A function that returns *something* no matter what is worse than one that fails loudly, because the failure ends up in your findings instead of your console.

### Part 1 — a fetch you can trust

**TODO:** finish `get_weather`. Requirements:

- sets a timeout
- retries on 429 and 5xx with an increasing delay
- raises immediately on 400 or 404 — those are your bug, not theirs
- returns a DataFrame from the `daily` block

You wrote this shape in Wednesday's studio. Rebuild it from memory if you can.

In [1]:
import requests, pandas as pd, time

BASE = 'https://archive-api.open-meteo.com/v1/archive'
RETRY_STATUSES = {429, 500, 502, 503, 504}

def get_weather(start, end, lat=38.03, lon=-78.51, tries=3):
    """Daily max temperature and total precipitation for a date range."""
    params = {
        'latitude': lat, 'longitude': lon,
        'start_date': start, 'end_date': end,
        'daily': 'temperature_2m_max,precipitation_sum',
        'timezone': 'America/New_York',
    }
    # TODO: loop up to `tries` times | verified
    # TODO:   request with a timeout; return pd.DataFrame(r.json()['daily']) on 200 |veriied
    # TODO:   raise on a status that is not in RETRY_STATUSES | verified
    # TODO:   otherwise sleep (increasing each time) and try again | verifeid
    # TODO: raise if every attempt failed | verified
    base_delay=0.01
    last = None
    #kiio up to tries times
    for attempt in range(tries):
        try:
            r = requests.get(BASE, params=params, timeout=15)
            #reqest w/ timeoutm "return r.json", "if == 200"
            if r.status_code == 200:
                return r.json()
                #NOT IN RETRY STATUSES
            if r.status_code not in RETRY_STATUSES:
                raise RuntimeError(
                    f'not retryable: {r.status_code} {r.text[:200]}')
            last = f'status {r.status_code}'
        except requests.RequestException as e:
            last = repr(e)
        delay = base_delay * (2 ** attempt)
        print(f'attempt {attempt + 1} failed ({last}); retrying in {delay}s')
        #sleeps when attempt is failed
        time.sleep(delay)

    raise RuntimeError(f'gave up after {tries} attempts: {last}')
    #prints error if failure after all tries




    pass

### Part 2 — two home weekends

Pull weather for **2026-09-05** and **2026-10-03**. One range covering both is fine and is one request instead of two — be nice to free APIs.

If the call genuinely fails, use the fallback in Part 3, but you must record that you did.

In [2]:
# TODO: call get_weather once for a range that spans both dates,
#       then keep just the two dates you care about
used_fallback = False

weather_df = get_weather(start='2026-09-05', end='2026-10-03')
#convert to a table where time could be accurately grabbed such that only the necessary dates are chosen
weather_df=pd.DataFrame(weather_df['daily'])

dates_df = weather_df[weather_df['time'].isin(['2026-09-05', '2026-10-03'])].copy()

print(dates_df)

          time  temperature_2m_max  precipitation_sum
0   2026-09-05                29.4                2.5
28  2026-10-03                22.0               14.7


### Part 3 — the fallback, labelled as a fallback

Sample values so the notebook still runs without a network. The important part is the `source` column: whoever reads your results can tell which rows are real.

In [3]:
FALLBACK = pd.DataFrame({
    'time': ['2026-09-05', '2026-10-03'],
    'temperature_2m_max': [30.1, 18.4],
    'precipitation_sum': [0.0, 22.5],
})

# TODO: if you used the fallback, set used_fallback = True and build `wx` from FALLBACK
# TODO: add a 'source' column to `wx` -- 'open-meteo' or 'fallback sample'

### Part 4 — make it joinable

**TODO:** produce a frame with a real datetime column named `date`, plus `temp_max`, `precip_mm`, and `source`. Rename the API's column names to yours — your pipeline should not inherit somebody else's naming.

In [3]:
# TODO

wx = dates_df.rename(columns={
    'time': 'date',
    'temperature_2m_max': 'temp_max',
    'precipitation_sum': 'precip_mm',}).copy()  #making a copy so that the originals do not get editied in the worst case scenario

wx['date'] = pd.to_datetime(wx['date'])
if used_fallback:
    wx['source'] = 'fallback'
else:
    wx['source'] = 'open-meteo'

### Part 5 — validate before you use it

This is the habit that separates a pipeline from a script. State what must be true, and find out here rather than three cells later inside a join.

**TODO:** uncomment and make them pass.

In [4]:
assert list(wx.columns) == ['date', 'temp_max', 'precip_mm', 'source']
assert len(wx) == 2, f'expected 2 rows, got {len(wx)}'
assert pd.api.types.is_datetime64_any_dtype(wx['date'])
assert wx['precip_mm'].notna().all(), 'missing precipitation'
assert (wx['precip_mm'] >= 0).all(), 'negative rainfall is not a thing'
print('weather table is safe to join.')

weather table is safe to join.


### Part 6 — answer the question

**TODO:** which of the two dates was wetter, and by how much? Print a sentence a vendor could read, not just a number.

In [6]:
# TODO

sep_5 = wx[wx['date'] == '2026-09-05'].iloc[0]
oct_3 = wx[wx['date'] == '2026-10-03'].iloc[0]

difference = abs(sep_5['precip_mm'] - oct_3['precip_mm'])

if sep_5['precip_mm'] > oct_3['precip_mm']:
    wetter_date = "September 5, 2026"
    drier_date = "October 3, 2026"
else:
    wetter_date = "October 3, 2026"
    drier_date = "September 5, 2026"

print(f"Between the two dates, {wetter_date} was wetter than {drier_date} by {difference:.1f} mm of precipitation.") # how to research and relearn again how to do decimal point limits in formatting demonstration

Between the two dates, October 3, 2026 was wetter than September 5, 2026 by 12.2 mm of precipitation.


### Part 7 — cite it

**TODO:** print the endpoint, the coordinates, the date range, when you pulled it, and whether the data is real or fallback.

In [15]:
# TODO

import datetime


print('Endpoint:', BASE)
print(f"Coordinates: Latitude 38.03, Longitude -78.51")
print(f"Date Range: 2026-09-05 to 2026-10-03")
print('Pulled: ', datetime.datetime.now().strftime('%Y-%m-%d %H:%M')) #required assistance of ai to discover a second datetime. call was necessary
print(f"Data Source Type: {'Fallback (Sample Data)' if used_fallback else 'Real (API)'}")

Endpoint: https://archive-api.open-meteo.com/v1/archive
Coordinates: Latitude 38.03, Longitude -78.51
Date Range: 2026-09-05 to 2026-10-03
Pulled:  2026-10-12 03:16
Data Source Type: Real (API)


### Write-up

Suppose `get_weather` returned an empty DataFrame on failure instead of raising. Walk through what happens next: you merge it onto 200 days of sales and chart rain against revenue. What does the chart show, and why would nobody catch it?

If the function returned an empty DataFrame on failure without an error notification or any kind then a left join against 200 days of sales data would keep the 200 sales rows information preserved just like how a left join should do, but all the data in the weather column would have NaN values and no data analysis can be done with no data.

Nobody catches it because it is good practice to drop rows with NaN since they are more likely to run into errors. So all 200 sales rows with NaN values would be dropped and not considered in analysis, flying under the radar because no error was raised.

### Stretch

Pull the same two dates for a second location — Orlando is at roughly `28.54, -81.38` — and put both cities in one long-format frame with a `city` column. That is the exact shape the midterm needs when you compare stores in different places.

In [19]:
# stretch

locations = {
    'Charlottesville': {'lat': 38.03, 'lon': -78.51},
    'Orlando': {'lat': 28.54, 'lon': -81.38}
}

frames = []
for city, coords in locations.items():
    try:
        response = get_weather('2026-09-05', '2026-10-03', lat=coords['lat'], lon=coords['lon'])
        #make a table again to properly look at rows of data
        city_df = pd.DataFrame(response['daily'])
        city_df['source'] = 'open-meteo'
    except Exception:
        #good practice to have a fallback
        city_df = pd.DataFrame({
            'time': ['2026-09-05', '2026-10-03'],
            'temperature_2m_max': [30.0, 28.0] if city == 'Orlando' else [28.5, 22.1],
            'precipitation_sum': [1.2, 0.0] if city == 'Orlando' else [0.0, 4.2]
        })
        city_df['source'] = 'fallback'

    city_df['city'] = city
    frames.append(city_df) # adds the generated city data frame to the originally empty list of data frames

df_combined = pd.concat(frames, ignore_index=True) # had to look this up
#fixing names of columns again as in above part
df_combined = df_combined.rename(columns={
    'time': 'date',
    'temperature_2m_max': 'temp_max',
    'precipitation_sum': 'precip_mm'
})
df_combined['date'] = pd.to_datetime(df_combined['date'])

# Filter down to the exact two dates
long_format_df = df_combined[df_combined['date'].isin(['2026-09-05', '2026-10-03'])].reset_index(drop=True)
print(long_format_df)

        date  temp_max  precip_mm      source             city
0 2026-09-05      29.4        2.5  open-meteo  Charlottesville
1 2026-10-03      22.0       14.7  open-meteo  Charlottesville
2 2026-09-05      32.1        3.9  open-meteo          Orlando
3 2026-10-03      30.4       14.5  open-meteo          Orlando


/tmp/ipykernel_12177/4278620229.py:37: FutureWarning: The behavior of 'isin' with dtype=datetime64[ns] and castable values (e.g. strings) is deprecated. In a future version, these will not be considered matching by isin. Explicitly cast to the appropriate dtype before calling isin instead.
  long_format_df = df_combined[df_combined['date'].isin(['2026-09-05', '2026-10-03'])].reset_index(drop=True)
